# PDF Chatbot

In [2]:
from langchain_community.document_loaders import PyPDFLoader

In [3]:
def pdf_loader(path):
    loader = PyPDFLoader(path)
    return loader.load()

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [6]:
def text_splitter(text):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1100,
        chunk_overlap=0,
    )
    return splitter.split_documents(text)

In [8]:
from langchain_huggingface import HuggingFaceEmbeddings

In [9]:
from langchain_community.vectorstores import Chroma

In [10]:
embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
def vector(chunks):
    vectorstore = Chroma(
        embedding_function = embeddings,
        persist_directory = 'chroma_db',
        collection_name = 'sample'
    )
    vectorstore.add_documents(chunks)
    return vectorstore

In [13]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

NameError: name 'vectorstore' is not defined

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.3
)

In [ ]:
def make_context(results):
    return "\n\n".join([doc.page_content for doc in results])

In [ ]:
from langchain_core.prompts import PromptTemplate

In [ ]:
template = PromptTemplate(
    input_variable=["context","query"],
    template = """You are an intelligent AI assistant.
                Use ONLY the information provided in the context below.
                Rules:
                    1. Answer only from the provided context.
                    2. If the answer is not available in the context, reply:
                           "I couldn't find that information in the provided document."
                    3. Do not make up information.
                    4. Keep the answer clear and well structured.
                Context: {context}
                Question: {query}
                Answer:"""
)

In [ ]:
prompt = template.format(
    context = context,
    query = query
)

In [ ]:
from langchain_core.output_parsers import StrOutputParser

In [ ]:
parser = StrOutputParser()

In [ ]:
from langchain_core.runnables import RunnableSequence, RunnablePassthrough, RunnableLambda, RunnableParallel

In [ ]:
chain1 = RunnableLambda(pdf_loader) | RunnableLambda(text_splitter) | RunnableLambda(vector)

In [ ]:
chain2 = retriever | RunnableLambda(make_context) | parser

In [ ]:
chain3 = RunnableParallel({
    "context" : chain2,
    "query" : RunnablePassthrough()
})

In [ ]:
chain4 = chain3 | template | llm | parser

In [172]:
vectorstore = chain1.invoke('The Hundred PageML.pdf')

In [173]:
ans = chain4.invoke("What is machine learning?")
print(ans)

Machine learning is a subfield of computer science concerned with building algorithms that rely on a collection of examples of some phenomenon to be useful.

It can also be defined as the process of solving a practical problem by:
1.  Gathering a dataset.
2.  Algorithmically building a statistical model based on that dataset, which is then used to solve the practical problem.

The terms "learning" and "machine learning" are used interchangeably. It is a universally recognized term that refers to the science and engineering of building machines capable of doing various useful things without being explicitly programmed to do so.


In [ ]:
for i, doc in enumerate(results, start=1):
    print("="*80)
    print(f"Chunk {i}")
    print("="*80)

    print(doc.page_content)

    print("\nMetadata:")
    print(doc.metadata)